# NER Notebook 3: End-to-End NER Pipeline

**BSAN 6200 — Text Mining & Social Media Analytics**

## What You'll Learn
1. **Load annotations** from Label Studio (JSON export)
2. **Convert** Label Studio JSON to IOB2 training format
3. **Quality check** annotation statistics and IOB consistency
4. **Train a CRF model** on your annotated data
5. **Evaluate** with entity-level metrics (seqeval)
6. **Export predictions** back to Label Studio for the active learning loop

## The Full Pipeline
```
Label Studio Export → Load JSON → Convert to IOB2 → Quality Check →
Split Data → Train Model → Evaluate (seqeval) →
Predict on New Text → Export to Label Studio → Human Review → Repeat
```

## Prerequisites
- Notebook 1 (NER inference, data formats, Label Studio I/O)
- Notebook 2 (CRF training, seqeval evaluation)

---

In [ ]:
#!pip install sklearn-crfsuite seqeval


In [10]:
# ============================================================
# SETUP: Install and import required packages
# ============================================================

import json
import re as regex_module
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
import sklearn_crfsuite
from seqeval.metrics import classification_report as seq_report
from seqeval.metrics import f1_score as seq_f1

print("All packages imported!")

All packages imported!


In [11]:
# ============================================================
# STEP 1: Load Label Studio JSON Export
# ============================================================
# Label Studio exports annotations as JSON files.
# Each task has: "data" (the text), "annotations" (list of annotation sets),
# and each annotation has "result" (list of entity spans).
#
# Label Studio JSON structure:
# [
#   {
#     "data": {"text": "Apple announced..."},
#     "annotations": [{
#       "result": [
#         {"value": {"start": 0, "end": 5, "text": "Apple", "labels": ["ORG"]},
#          "from_name": "label", "to_name": "text", "type": "labels"}
#       ]
#     }]
#   },
#   ...
# ]

import json
import re as regex_module
from collections import Counter

# Create sample data simulating a Label Studio export
# In your project, replace this with your actual Label Studio JSON export
sample_tasks = [
    {"data": {"text": "Apple announced a new MacBook Pro at their Cupertino headquarters."},
     "annotations": [{"result": [
         {"value": {"start": 0, "end": 5, "text": "Apple", "labels": ["ORG"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 22, "end": 33, "text": "MacBook Pro", "labels": ["PRODUCT"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 43, "end": 62, "text": "Cupertino headquarters", "labels": ["LOC"]}, "from_name": "label", "to_name": "text", "type": "labels"},
     ]}]},
    {"data": {"text": "Elon Musk is the CEO of Tesla and SpaceX."},
     "annotations": [{"result": [
         {"value": {"start": 0, "end": 9, "text": "Elon Musk", "labels": ["PER"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 24, "end": 29, "text": "Tesla", "labels": ["ORG"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 34, "end": 40, "text": "SpaceX", "labels": ["ORG"]}, "from_name": "label", "to_name": "text", "type": "labels"},
     ]}]},
    {"data": {"text": "The S&P 500 rose 2% after the Federal Reserve cut rates in December."},
     "annotations": [{"result": [
         {"value": {"start": 4, "end": 11, "text": "S&P 500", "labels": ["PRODUCT"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 30, "end": 45, "text": "Federal Reserve", "labels": ["ORG"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 62, "end": 70, "text": "December", "labels": ["DATE"]}, "from_name": "label", "to_name": "text", "type": "labels"},
     ]}]},
    {"data": {"text": "Microsoft Azure revenue grew 29% to $24.3 billion in Q2 2024."},
     "annotations": [{"result": [
         {"value": {"start": 0, "end": 15, "text": "Microsoft Azure", "labels": ["PRODUCT"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 36, "end": 49, "text": "$24.3 billion", "labels": ["MONEY"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 53, "end": 60, "text": "Q2 2024", "labels": ["DATE"]}, "from_name": "label", "to_name": "text", "type": "labels"},
     ]}]},
    {"data": {"text": "Goldman Sachs analyst Sarah Chen upgraded Nvidia to buy."},
     "annotations": [{"result": [
         {"value": {"start": 0, "end": 13, "text": "Goldman Sachs", "labels": ["ORG"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 22, "end": 31, "text": "Sarah Chen", "labels": ["PER"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 41, "end": 47, "text": "Nvidia", "labels": ["ORG"]}, "from_name": "label", "to_name": "text", "type": "labels"},
     ]}]},
    {"data": {"text": "Jeff Bezos founded Amazon in his garage in Seattle."},
     "annotations": [{"result": [
         {"value": {"start": 0, "end": 10, "text": "Jeff Bezos", "labels": ["PER"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 19, "end": 25, "text": "Amazon", "labels": ["ORG"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 43, "end": 50, "text": "Seattle", "labels": ["LOC"]}, "from_name": "label", "to_name": "text", "type": "labels"},
     ]}]},
    {"data": {"text": "Google released Gemini Pro to compete with GPT-4 from OpenAI."},
     "annotations": [{"result": [
         {"value": {"start": 0, "end": 6, "text": "Google", "labels": ["ORG"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 16, "end": 26, "text": "Gemini Pro", "labels": ["PRODUCT"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 44, "end": 49, "text": "GPT-4", "labels": ["PRODUCT"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 55, "end": 61, "text": "OpenAI", "labels": ["ORG"]}, "from_name": "label", "to_name": "text", "type": "labels"},
     ]}]},
    {"data": {"text": "Warren Buffett sold $6.2 billion in Bank of America shares in August 2024."},
     "annotations": [{"result": [
         {"value": {"start": 0, "end": 14, "text": "Warren Buffett", "labels": ["PER"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 20, "end": 32, "text": "$6.2 billion", "labels": ["MONEY"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 36, "end": 52, "text": "Bank of America", "labels": ["ORG"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 63, "end": 74, "text": "August 2024", "labels": ["DATE"]}, "from_name": "label", "to_name": "text", "type": "labels"},
     ]}]},
    {"data": {"text": "Netflix reported 13 million new subscribers in Q4."},
     "annotations": [{"result": [
         {"value": {"start": 0, "end": 7, "text": "Netflix", "labels": ["ORG"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 46, "end": 48, "text": "Q4", "labels": ["DATE"]}, "from_name": "label", "to_name": "text", "type": "labels"},
     ]}]},
    {"data": {"text": "Sam Altman presented GPT-5 at the OpenAI DevDay in San Francisco."},
     "annotations": [{"result": [
         {"value": {"start": 0, "end": 10, "text": "Sam Altman", "labels": ["PER"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 21, "end": 26, "text": "GPT-5", "labels": ["PRODUCT"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 34, "end": 40, "text": "OpenAI", "labels": ["ORG"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 51, "end": 64, "text": "San Francisco", "labels": ["LOC"]}, "from_name": "label", "to_name": "text", "type": "labels"},
     ]}]},
    {"data": {"text": "Toyota Motor Company reported record profits of $28 billion for fiscal year 2024."},
     "annotations": [{"result": [
         {"value": {"start": 0, "end": 21, "text": "Toyota Motor Company", "labels": ["ORG"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 49, "end": 60, "text": "$28 billion", "labels": ["MONEY"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 65, "end": 81, "text": "fiscal year 2024", "labels": ["DATE"]}, "from_name": "label", "to_name": "text", "type": "labels"},
     ]}]},
    {"data": {"text": "The iPhone 15 became the best selling smartphone in North America."},
     "annotations": [{"result": [
         {"value": {"start": 4, "end": 13, "text": "iPhone 15", "labels": ["PRODUCT"]}, "from_name": "label", "to_name": "text", "type": "labels"},
         {"value": {"start": 50, "end": 63, "text": "North America", "labels": ["LOC"]}, "from_name": "label", "to_name": "text", "type": "labels"},
     ]}]},
]

# Save as JSON file (simulating Label Studio export)
with open("label_studio_export.json", "w") as f:
    json.dump(sample_tasks, f, indent=2)

print(f"Loaded {len(sample_tasks)} annotated tasks from Label Studio export")
print(f"\nExample task structure:")
print(json.dumps(sample_tasks[0], indent=2)[:400] + "...")

Loaded 12 annotated tasks from Label Studio export

Example task structure:
{
  "data": {
    "text": "Apple announced a new MacBook Pro at their Cupertino headquarters."
  },
  "annotations": [
    {
      "result": [
        {
          "value": {
            "start": 0,
            "end": 5,
            "text": "Apple",
            "labels": [
              "ORG"
            ]
          },
          "from_name": "label",
          "to_name": "text",
          "type": "...


In [12]:
# ============================================================
# STEP 2: Convert Label Studio JSON → IOB2 Format
# ============================================================

def label_studio_to_iob2(ls_data):
    """
    Convert Label Studio JSON export to IOB2 token-label pairs.

    Handles the character-to-token alignment:
    - Tokenize by whitespace (preserving character positions)
    - Map character-level entity spans to token-level IOB tags
    - B- prefix for first token, I- prefix for continuation tokens

    Args:
        ls_data: list of Label Studio task dicts (parsed JSON)

    Returns:
        List of sentences, where each sentence is a list of (token, tag) tuples
    """
    all_sentences = []

    for task in ls_data:
        text = task["data"]["text"]

        # Get annotations — Label Studio uses "annotations" for completed,
        # "predictions" for pre-annotations
        annotations = task.get("annotations", task.get("predictions", []))
        if not annotations:
            continue

        # Get entity spans from the first annotation set
        results = annotations[0].get("result", [])
        entities = []
        for r in results:
            if r.get("type") == "labels":
                val = r["value"]
                entities.append((val["start"], val["end"], val["labels"][0]))

        entities.sort(key=lambda x: x[0])

        # Tokenize and track character positions
        tokens = []
        token_spans = []
        for match in regex_module.finditer(r'\S+', text):
            tokens.append(match.group())
            token_spans.append((match.start(), match.end()))

        # Initialize all tags as "O"
        tags = ["O"] * len(tokens)

        # Assign IOB tags based on character spans
        for ent_start, ent_end, label in entities:
            entity_started = False
            for i, (tok_start, tok_end) in enumerate(token_spans):
                if tok_start >= ent_start and tok_start < ent_end:
                    if not entity_started:
                        tags[i] = f"B-{label}"
                        entity_started = True
                    else:
                        tags[i] = f"I-{label}"

        all_sentences.append(list(zip(tokens, tags)))

    return all_sentences

# Convert
with open("label_studio_export.json") as f:
    ls_data = json.load(f)

sentences = label_studio_to_iob2(ls_data)

# Display first 3 examples
for i, sent in enumerate(sentences[:3]):
    print(f"\nSentence {i+1}:")
    for token, tag in sent:
        marker = " ←" if tag != "O" else ""
        print(f"  {token:<25} {tag}{marker}")


Sentence 1:
  Apple                     B-ORG ←
  announced                 O
  a                         O
  new                       O
  MacBook                   B-PRODUCT ←
  Pro                       I-PRODUCT ←
  at                        O
  their                     O
  Cupertino                 B-LOC ←
  headquarters.             I-LOC ←

Sentence 2:
  Elon                      B-PER ←
  Musk                      I-PER ←
  is                        O
  the                       O
  CEO                       O
  of                        O
  Tesla                     B-ORG ←
  and                       O
  SpaceX.                   B-ORG ←

Sentence 3:
  The                       O
  S&P                       B-PRODUCT ←
  500                       I-PRODUCT ←
  rose                      O
  2%                        O
  after                     O
  the                       O
  Federal                   B-ORG ←
  Reserve                   I-ORG ←
  cut                      

In [13]:
# ============================================================
# STEP 3: Quality Check & Statistics
# ============================================================
# Always verify your data before training!

all_tags = [tag for sent in sentences for _, tag in sent]
entity_tags = [tag for tag in all_tags if tag != "O"]
entity_types = Counter(tag.split("-", 1)[1] for tag in entity_tags if "-" in tag)

print("ANNOTATION STATISTICS")
print("=" * 50)
print(f"Total sentences:    {len(sentences)}")
print(f"Total tokens:       {len(all_tags)}")
print(f"Entity tokens:      {len(entity_tags)} ({len(entity_tags)/len(all_tags)*100:.1f}%)")
print(f"O tokens:           {len(all_tags) - len(entity_tags)} ({(len(all_tags)-len(entity_tags))/len(all_tags)*100:.1f}%)")
print(f"\nEntity type distribution:")
for etype, count in entity_types.most_common():
    print(f"  {etype:<15} {count:>4} tokens")

# IOB consistency check
issues = []
for i, sent in enumerate(sentences):
    tags = [tag for _, tag in sent]
    for j, tag in enumerate(tags):
        if tag.startswith("I-"):
            if j == 0 or (not tags[j-1].endswith(tag[2:]) or tags[j-1] == "O"):
                issues.append(f"Sentence {i}, token {j}: I-tag without B-tag: {tag}")

if issues:
    print(f"\n⚠ {len(issues)} IOB consistency issues found!")
    for issue in issues[:3]:
        print(f"  {issue}")
else:
    print(f"\n✓ All IOB tags are consistent!")

ANNOTATION STATISTICS
Total sentences:    12
Total tokens:       127
Entity tokens:      57 (44.9%)
O tokens:           70 (55.1%)

Entity type distribution:
  ORG               18 tokens
  PRODUCT           11 tokens
  PER               10 tokens
  LOC                7 tokens
  DATE               6 tokens
  MONEY              5 tokens

✓ All IOB tags are consistent!


In [14]:
# ============================================================
# STEP 4: Train/Dev/Test Split
# ============================================================
# Split annotated data for model training and evaluation.
# 70% train, 30% test (with small datasets, keep it simple)

from sklearn.model_selection import train_test_split

train_sents, test_sents = train_test_split(
    sentences, test_size=0.3, random_state=42
)

print(f"Train: {len(train_sents)} sentences")
print(f"Test:  {len(test_sents)} sentences")

Train: 8 sentences
Test:  4 sentences


---
## Step 5: Train CRF Model

We reuse the CRF approach from Notebook 2. The feature extraction
functions define what the model sees for each token: word shape,
suffixes, and neighboring word context.

In [15]:
# ============================================================
# STEP 4: Train CRF on Annotations
# ============================================================
import sklearn_crfsuite
from seqeval.metrics import classification_report as seq_report
from sklearn.model_selection import train_test_split

# Reuse the feature extraction functions from Notebook 2
def word2features(sent, i):
    word = sent[i][0]
    features = {
        'bias': 1.0, 'word.lower()': word.lower(),
        'word[-3:]': word[-3:], 'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(), 'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(), 'word.isalpha()': word.isalpha(),
        'word.length': len(word), 'word.has_hyphen': '-' in word,
        'word.has_dollar': '$' in word,
    }
    if i > 0:
        w = sent[i-1][0]
        features.update({'-1:word.lower()': w.lower(), '-1:word.istitle()': w.istitle()})
    else:
        features['BOS'] = True
    if i < len(sent) - 1:
        w = sent[i+1][0]
        features.update({'+1:word.lower()': w.lower(), '+1:word.istitle()': w.istitle()})
    else:
        features['EOS'] = True
    return features

sent2features = lambda s: [word2features(s, i) for i in range(len(s))]
sent2labels = lambda s: [label for _, label in s]
sent2tokens = lambda s: [token for token, _ in s]

# Split: 70% train, 30% test
train_sents, test_sents = train_test_split(sentences, test_size=0.3, random_state=42)

X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]
X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

# Train CRF
crf = sklearn_crfsuite.CRF(algorithm='lbfgs', c1=0.1, c2=0.1,
                            max_iterations=100, all_possible_transitions=True)
crf.fit(X_train, y_train)

# Evaluate
y_pred = crf.predict(X_test)
print("CRF Entity-Level Report:")
print("=" * 50)
print(seq_report(y_test, y_pred))

CRF Entity-Level Report:
              precision    recall  f1-score   support

        DATE       1.00      0.50      0.67         2
         LOC       0.00      0.00      0.00         2
       MONEY       0.00      0.00      0.00         1
         ORG       0.67      0.50      0.57         4
         PER       0.50      1.00      0.67         1
     PRODUCT       0.00      0.00      0.00         2

   micro avg       0.67      0.33      0.44        12
   macro avg       0.36      0.33      0.32        12
weighted avg       0.43      0.33      0.36        12



In [16]:
# ============================================================
# STEP 6: Detailed Evaluation
# ============================================================
# Entity-level evaluation with seqeval (the correct way)
# Plus per-entity-type breakdown

y_pred = crf.predict(X_test)

print("ENTITY-LEVEL EVALUATION (seqeval)")
print("=" * 60)
print(seq_report(y_test, y_pred))

overall_f1 = seq_f1(y_test, y_pred)
print(f"Overall F1: {overall_f1:.4f}")

# ---- Quick error inspection ----
# Show sentences where the model made mistakes
print("\n\nSAMPLE ERRORS (first 3 mismatches):")
print("=" * 60)
error_count = 0
for sent, true_tags, pred_tags in zip(test_sents, y_test, y_pred):
    tokens = [t for t, _ in sent]
    if true_tags != pred_tags and error_count < 3:
        print(f"\n  Text: {' '.join(tokens)}")
        for tok, t, p in zip(tokens, true_tags, pred_tags):
            if t != p:
                print(f"    {tok:<20} True: {t:<12} Pred: {p}")
        error_count += 1

ENTITY-LEVEL EVALUATION (seqeval)
              precision    recall  f1-score   support

        DATE       1.00      0.50      0.67         2
         LOC       0.00      0.00      0.00         2
       MONEY       0.00      0.00      0.00         1
         ORG       0.67      0.50      0.57         4
         PER       0.50      1.00      0.67         1
     PRODUCT       0.00      0.00      0.00         2

   micro avg       0.67      0.33      0.44        12
   macro avg       0.36      0.33      0.32        12
weighted avg       0.43      0.33      0.36        12

Overall F1: 0.4444


SAMPLE ERRORS (first 3 mismatches):

  Text: Toyota Motor Company reported record profits of $28 billion for fiscal year 2024.
    Toyota               True: B-ORG        Pred: B-PER
    Motor                True: I-ORG        Pred: I-PER
    Company              True: I-ORG        Pred: O
    billion              True: B-MONEY      Pred: O
    year                 True: B-DATE       Pred: O
    202

---
## Step 7: Inference on New Text

Use the trained CRF to predict entities on unseen text.
This is also useful for checking model behavior qualitatively.

In [17]:
# ============================================================
# STEP 7: Run CRF Inference on New Sentences
# ============================================================

def predict_ner(text, crf_model):
    """
    Run NER inference on a single text string.

    Args:
        text: raw text string
        crf_model: trained CRF model

    Returns:
        list of (token, predicted_tag) tuples
    """
    tokens = text.split()
    pseudo_sent = [(tok, "O") for tok in tokens]
    features = sent2features(pseudo_sent)
    pred_tags = crf_model.predict([features])[0]
    return list(zip(tokens, pred_tags))

# Test on new sentences
new_texts = [
    "Meta Platforms invested $10 billion in AI research in 2024.",
    "Sundar Pichai announced new Pixel features at Google I/O in Mountain View.",
    "The Dow Jones dropped 500 points after the Fed announcement on Wednesday.",
]

for text in new_texts:
    print(f"\nText: {text}")
    result = predict_ner(text, crf)
    entities = [(tok, tag) for tok, tag in result if tag != "O"]
    if entities:
        for tok, tag in entities:
            print(f"  → {tok:<25} {tag}")
    else:
        print("  → No entities found")


Text: Meta Platforms invested $10 billion in AI research in 2024.
  → Meta                      B-PER
  → Platforms                 I-PER
  → $10                       B-MONEY
  → billion                   I-MONEY

Text: Sundar Pichai announced new Pixel features at Google I/O in Mountain View.
  → Sundar                    B-PER
  → Pichai                    I-PER
  → Google                    B-ORG
  → I/O                       I-ORG
  → View.                     B-ORG

Text: The Dow Jones dropped 500 points after the Fed announcement on Wednesday.
  → Dow                       B-PRODUCT
  → Jones                     I-PRODUCT
  → Wednesday.                B-ORG


---
## Step 8: Export Predictions for Label Studio Review

This closes the **active learning loop**: model predictions become
pre-annotations in Label Studio for human review and correction.

The predictions go into the `"predictions"` field (not `"annotations"`)
so Label Studio renders them as editable suggestions.

In [18]:
# ============================================================
# STEP 5: Predict on New Data & Export for Label Studio Review
# ============================================================
# This is the ACTIVE LEARNING step:
# Use model predictions to pre-annotate new data → import to
# Label Studio as predictions → human review and correction

def predict_and_export_label_studio(new_texts, crf_model, output_file):
    """
    Run CRF predictions on new texts and export as Label Studio JSON.

    This creates pre-annotated tasks that can be imported into
    Label Studio for human review — much faster than annotating from scratch!

    The key: we put predictions in the "predictions" field (not "annotations")
    so Label Studio shows them as suggestions, not confirmed labels.

    Args:
        new_texts: List of raw text strings
        crf_model: Trained CRF model
        output_file: Path for JSON output
    """
    tasks = []
    for text in new_texts:
        # Tokenize and extract features
        tokens = text.split()
        pseudo_sent = [(tok, "O") for tok in tokens]
        features = sent2features(pseudo_sent)

        # Predict tags
        pred_tags = crf_model.predict([features])[0]

        # Convert IOB tags to Label Studio format (character spans)
        results = []
        char_pos = 0
        i = 0
        while i < len(tokens):
            start = text.find(tokens[i], char_pos)

            if pred_tags[i].startswith("B-"):
                entity_label = pred_tags[i][2:]
                entity_start = start
                entity_end = start + len(tokens[i])

                # Extend through continuation (I-) tags
                j = i + 1
                while j < len(tokens) and pred_tags[j] == f"I-{entity_label}":
                    next_start = text.find(tokens[j], entity_end)
                    entity_end = next_start + len(tokens[j])
                    j += 1

                entity_text = text[entity_start:entity_end]
                results.append({
                    "from_name": "label",
                    "to_name": "text",
                    "type": "labels",
                    "value": {
                        "start": entity_start,
                        "end": entity_end,
                        "text": entity_text,
                        "labels": [entity_label],
                    },
                })
                char_pos = entity_end
                i = j
            else:
                char_pos = start + len(tokens[i])
                i += 1

        task = {
            "data": {"text": text},
            "predictions": [{"result": results}] if results else [],
        }
        tasks.append(task)

    with open(output_file, "w") as f:
        json.dump(tasks, f, indent=2)

    print(f"Exported {len(tasks)} pre-annotated tasks to {output_file}")
    print("Import this file into Label Studio for human review!")

# New texts to pre-annotate
new_texts = [
    "Meta Platforms invested $10 billion in AI research in 2024.",
    "Sundar Pichai announced new Pixel features at Google I/O in Mountain View.",
    "The Dow Jones dropped 500 points after the Fed announcement on Wednesday.",
    "Amazon's AWS division hired 5,000 new engineers in India last quarter.",
    "Berkshire Hathaway reported $97 billion in revenue for fiscal year 2023.",
]

predict_and_export_label_studio(new_texts, crf, "predictions_for_label_studio.json")

# Show what was exported
print("\n--- Exported JSON (import into Label Studio for review) ---")
with open("predictions_for_label_studio.json") as f:
    data = json.load(f)
    for task in data[:2]:
        print(f"\nText: {task['data']['text'][:55]}...")
        preds = task.get('predictions', [{}])
        if preds and preds[0].get('result'):
            for r in preds[0]['result']:
                v = r['value']
                print(f"  → {v['text']:<25} {v['labels'][0]}")
        else:
            print("  → No entities predicted")

Exported 5 pre-annotated tasks to predictions_for_label_studio.json
Import this file into Label Studio for human review!

--- Exported JSON (import into Label Studio for review) ---

Text: Meta Platforms invested $10 billion in AI research in 2...
  → Meta Platforms            PER
  → $10 billion               MONEY

Text: Sundar Pichai announced new Pixel features at Google I/...
  → Sundar Pichai             PER
  → Google I/O                ORG
  → View.                     ORG


---
## The Active Learning Loop

This is how production NER systems continuously improve:

```
┌─────────────┐     ┌──────────────┐     ┌──────────────┐
│  New Data    │────▶│ Model Predicts│────▶│  Export to    │
│  (unlabeled) │     │ (pre-annotate)│     │ Label Studio  │
└─────────────┘     └──────────────┘     └──────┬───────┘
                                                 │
                    ┌──────────────┐     ┌───────▼──────┐
                    │  Retrain     │◀────│  Human Review │
                    │  Model       │     │  & Correction │
                    └──────┬───────┘     └──────────────┘
                           │
                    ┌──────▼───────┐
                    │  Evaluate    │
                    │  (seqeval)   │
                    └──────────────┘
```

### Practical Tips
1. **Start small**: 50-100 annotated sentences → train CRF → evaluate
2. **Pre-annotate**: Use model or LLM predictions → export as Label Studio JSON with "predictions" field
3. **Review & correct** in Label Studio (fix what the model got wrong)
4. **Track F1 across iterations** — stop when improvement plateaus
5. **Graduate to Transformers** when you have 200+ annotated sentences

---
## Complete NER Project Checklist

| Step | Tool | Output |
|------|------|--------|
| 1. Obtain data | Web scraping, APIs, docs | Raw text files |
| 2. Define entities | Annotation guideline (Word doc) | Entity definitions |
| 3. Pre-annotate (optional) | Pretrained model (spaCy, HF) | Label Studio JSON (predictions) |
| 4. Annotate | Label Studio (primary) or Doccano (backup) | Reviewed JSON export |
| 5. Quality check | This notebook (Step 3) | Clean IOB2 dataset |
| 6. Train | CRF (small data) or Transformer (Notebook 2) | Trained model |
| 7. Evaluate | seqeval entity-level F1 | Classification report |
| 8. Error analysis | Notebook 2 techniques | Error categories |
| 9. Export predictions | This notebook (Step 8) | Pre-annotated JSON |
| 10. Iterate | Import to Label Studio → review → retrain | Improved model |

